# Look at the errors of the SPDI column after validation
- output path: /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/spdi_verification

In [3]:
import ast
from importlib import reload
import pandas as pd
import sys
import os
import yaml

sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [ ]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'

In [28]:
# Link to the metadata output
directory_path = config['final_output_dir'] # '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format'
metadata_file_ending = '.metadata.tsv.gz'

# error directory
validation_dir = '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/spdi_verification'

## Definition of the indels in the metadata file
### Process: 
- combine the metadata files
- filter for indel

In [76]:
# helpful functions
def list_metadata_files_in_subdirectories(directory, file_ending, file_beginning="........."):
    """Returns all the files in the subdirectories of a given directory"""
    file_list = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file_ending in file and (file.startswith(file_beginning) or file_beginning == "........."):
                file_list.append(os.path.join(root, file))
    return file_list


def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x


def get_category(header):
    """
    Get the category of the headers set all to element if not scarmbled
    """

    label = hf.get_label(header)

    if 'scramble' in header:
        # Check if header is scrambled
        # Info: scrambled is in C_negative_neuron_NP and scramble MK
        # Note: Other cases are not checked with this function
        return 'scrambled'
    else:
        return 'element'


def get_reference_genome(row):
    """If col_category is synthetic or scrambled or in dnase_control_groups, set ref to GRCh37 else GRCh38"""
    row[col_ref] = 'GRCh38'
    return row


def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[-1] if match else None

import requests

def get_rsid_position(rsid, assembly="GRCh38"):
    url = f"https://rest.ensembl.org/variation/human/{rsid}"
    headers = {"Content-Type": "application/json"}
    response = requests.get(url, headers=headers, params={"genome": assembly})
    if response.status_code == 200:
        data = response.json()
        for mapping in data.get("mappings", []):
            if mapping["assembly_name"] == assembly:
                return {
                    "chromosome": mapping["seq_region_name"],
                    "start": mapping["start"],
                    "end": mapping["end"],
                }
    return None



# dict of chr number to refseq chromosome number
chrom_2_refseq = {
    "chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) # I think everything is now 0-based
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi(row):
    """
    Returns the SPDI identifier for the given variant using start and variant_pos
    Assumption: allele need to be set beforehands
    """
    if not (hf.is_alternative(row[col_allele])) and (row[my_col_ref_base] and row[my_col_alt_base]):
        row['SPDI'] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = f'{row[col_chr]}-{row[col_start]+row[col_variant_pos]}-{row[my_col_ref_base]}-{row[my_col_alt_base]}'
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row

def is_indel(variant_class):
    if isinstance(variant_class, list):
        return 'indel' in variant_class
    else: # variant class is just a string
        return variant_class == 'indel'

def is_non_zero_file(fpath):
    return os.path.isfile(fpath) and os.path.getsize(fpath) > 0

def read_and_eval_metadata(file):
    list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

    # load metadata file
    metadata_file = pd.read_csv(file, sep="\t")
    # Apply the safe_eval function to the specified columns
    for col in list_columns:
        metadata_file[col] = metadata_file[col].apply(safe_eval)
    return metadata_file

### Check for the overall numbers

In [23]:
file_list = list_metadata_files_in_subdirectories(directory_path, metadata_file_ending)
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]


n_indels = 0
for file in file_list:
    group_name = file.split('/')[-1].split(metadata_file_ending)[0]
    # load metadata file
    metadata_file = pd.read_csv(file, sep="\t")
    # Apply the safe_eval function to the specified columns
    for col in list_columns:
        metadata_file[col] = metadata_file[col].apply(safe_eval)
    # find the indel rows
    indel_rows = metadata_file.loc[metadata_file[col_variant_class].apply(is_indel)]
    if indel_rows.shape[0] > 0:
        # print(group_name)
        print(file)
        # print('n_indel', indel_rows.shape[0])
    n_indels += indel_rows.shape[0]

print(n_indels)


# GC_Mendelian_variants
# n_indel 6
# GC_Mohlke
# n_indel 1
# C_positive_heart_CAD
# n_indel 2

/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/GC_Mendelian_variants/GC_Mendelian_variants.metadata.tsv.gz
/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/GC_Mohlke/GC_Mohlke.metadata.tsv.gz
/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz
9


### Investigate them by hand:

In [82]:
file_list = list_metadata_files_in_subdirectories(directory_path, metadata_file_ending)
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]


n_indels = 0
# file_list = ['/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/cardiac_neuro_cava_random/cardiac_neuro_cava_random.metadata.tsv.gz']
for file in file_list:
    group_name = file.split('/')[-1].split(metadata_file_ending)[0]
    # load metadata file
    metadata_file = pd.read_csv(file, sep="\t")
    # Apply the safe_eval function to the specified columns
    for col in list_columns:
        metadata_file[col] = metadata_file[col].apply(safe_eval)
    # find the indel rows
    indel_rows = metadata_file.loc[metadata_file[col_variant_class].apply(is_indel)].copy()
    if indel_rows.shape[0] > 0:
        print(group_name)
        print(indel_rows[col_name])
        break
    n_indels += indel_rows.shape[0]

print(n_indels)

GC_Mendelian_variants
42     GC_Mendelian_variants:ALT_chr8:11703860G>T|GAT...
43     GC_Mendelian_variants:ALT_chr8:11703890AG>A|GA...
50     GC_Mendelian_variants:ALT_chr1:209816133C>CA|I...
98     GC_Mendelian_variants:ALT_chr7:156791255G>C|SH...
99     GC_Mendelian_variants:ALT_chr7:156791257G>A|SH...
100    GC_Mendelian_variants:ALT_chr7:156791274T>TTAA...
Name: name, dtype: object
0


In [21]:
indel_rows

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
42,GC_Mendelian_variants:ALT_chr8:11703860G>T|GAT...,CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr8,11703724,11703994,.,indel,165.0,NC_000008.11:11703889:AG:A,alt,NaN
43,GC_Mendelian_variants:ALT_chr8:11703890AG>A|GA...,CCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr8,11703754,11704024,.,indel,135.0,NC_000008.11:11703889:AG:A,alt,NaN
50,GC_Mendelian_variants:ALT_chr1:209816133C>CA|I...,TTGAGCCCAGGGGCTGAATCTGGAGCTTTGGGGCCTGGGAACCTCT...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,209815997,209816267,.,indel,135.0,NC_000001.11:209816132:C:CA,alt,NaN
98,GC_Mendelian_variants:ALT_chr7:156791255G>C|SH...,GAGATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr7,156791119,156791389,.,indel,154.0,NC_000007.14:156791273:T:TTAAGGAAGTGATT,alt,NaN
99,GC_Mendelian_variants:ALT_chr7:156791257G>A|SH...,GATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATGAC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr7,156791121,156791391,.,indel,152.0,NC_000007.14:156791273:T:TTAAGGAAGTGATT,alt,NaN
100,GC_Mendelian_variants:ALT_chr7:156791274T>TTAA...,TGTAATAAACACTAAGATCAAAACATGACCCAAGTTAAATTTCCTT...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr7,156791138,156791408,.,indel,135.0,NC_000007.14:156791273:T:TTAAGGAAGTGATT,alt,NaN


## Look at the SNP cases

In [81]:
file_list = list_metadata_files_in_subdirectories(validation_dir, file_ending='.tsv', file_beginning='error_')
metadata_files = list_metadata_files_in_subdirectories(directory_path, file_ending=metadata_file_ending)

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

ignorable_error_dfs = ['cardiac_neuro_cava_random']


for file in file_list:
    group_name = file.split('/')[-1].split(metadata_file_ending)[0].split('.tsv')[0].split('error_SPDI_')[1]
    # load file only if not empty:
    if not is_non_zero_file(file):
        continue
    if group_name in ignorable_error_dfs:
        continue
    error_file = pd.read_csv(file, sep="\t", header=None)
    error_file.columns = [col_SPDI, 'error_info', 'error_valid']

    if error_file.shape[0] > 0:
        print(group_name, f'n={error_file.shape[0]}')



C_positive_heart_CAD n=1
GC_Mendelian_variants n=3
GC_Mohlke n=1


In [128]:
file_list = list_metadata_files_in_subdirectories(validation_dir, file_ending='.tsv', file_beginning='error_')
metadata_files = list_metadata_files_in_subdirectories(directory_path, file_ending=metadata_file_ending)

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

ignorable_error_dfs = ['cardiac_neuro_cava_random']


for file in file_list:
    group_name = file.split('/')[-1].split(metadata_file_ending)[0].split('.tsv')[0].split('error_SPDI_')[1]
    # load file only if not empty:
    if not is_non_zero_file(file):
        continue
    if group_name in ignorable_error_dfs:
        continue
    error_file = pd.read_csv(file, sep="\t", header=None)
    error_file.columns = [col_SPDI, 'error_info', 'error_valid']

    if error_file.shape[0] > 0:
        print(group_name)
        print(file)
        # print(error_file.head())

        current_metadata_file = [file for file in metadata_files if group_name in file.split('/')[-1]]
        print(current_metadata_file)
        error_spdi_rows = []
        # get spdis in error df
        error_spdi_list = error_file[error_file.columns[0]].to_list()
        # concatenate metadata files:
        current_metadata = pd.concat([read_and_eval_metadata(file) for file in current_metadata_file], ignore_index=True)
        # print(current_metadata.head())
        current_metadata_alt = current_metadata.loc[current_metadata[col_allele].apply(hf.is_alternative)].copy()

        # print('metadata size:', current_metadata_alt.shape[0])
        # print('metadata size:', current_metadata_alt[col_SPDI].nunique())

        # print(current_metadata_alt.head())
        # print(current_metadata_alt[col_SPDI].to_list())
        current_metadata_alt[col_SPDI] = current_metadata[col_SPDI].astype(str)
        error_file[col_SPDI] = error_file[col_SPDI].astype(str)
        # filter df for error spdis (left join)

        result = pd.merge(error_file, current_metadata_alt, on=[col_SPDI], how="inner")
        result['group_name'] = group_name
        print(result.shape[0])
        # print(result.head())
        # print start and end of the metadata => create bed file => get reference sequences => find position of indel
        # store group_name_indel.bed
        # result[[col_chr, col_start, col_end, col_SPDI, 'group_name']].to_csv(f'{group_name}_indel.bed', index=False, sep="\t", header=None)

C_positive_heart_CAD
/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/spdi_verification/error_SPDI_C_positive_heart_CAD.tsv
['/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz']
1
GC_Mendelian_variants
/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/spdi_verification/error_SPDI_GC_Mendelian_variants.tsv
['/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/GC_Mendelian_variants/GC_Mendelian_variants.metadata.tsv.gz']
5
GC_Mohlke
/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/spdi_verification/error_SPDI_GC_Mohlke.tsv
['/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/GC_Mohlke/GC_Mohlke.metadata.tsv.gz']
1


- NC_000008.11:11703889:AG:A	NC_000008.11:11703890:GGGGGGG:GGGGGG
- ref: CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCC
- alt: CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCC
- => 165 0-based (start: 11703724) => 11703889
- NC_000008.11:11703889:AG:A	NC_000008.11:11703890:GGGGGGG:GGGGGG
- alt: CCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGA
- NC_000008.11:11703889:AG:A	NC_000008.11:11703890:GGGGGGG:GGGGGG
- alt: CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCC
- NC_000008.11:11703889:AG:A	NC_000008.11:11703890:GGGGGGG:GGGGGG
- alt: CCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGA
- NC_000001.11:209816132:C:CA	NC_000001.11:209816133:AA:AAA
- ref: TTGAGCCCAGGGGCTGAATCTGGAGCTTTGGGGCCTGGGAACCTCTCTACCTGCGTCAATGTCTGGAGGCCCTGAGAGTTTCGCTCAGGCTCAGAGCAGGCATCGCAACCTCCCAGTTACTATTCTGTGCTGTGGCAAGTGCCAGCTTGTCCTCTCTTCCCCACCCAGCCCGGGAAACCGGCAGCATTTCTAGTTCAGGCCCAGACCCGTCCTGGCAGCCTGGATTCCACTGCCTAGGCAGGAAGCTCATCTCAGCCCAGTGACCTTTTC
- alt: TTGAGCCCAGGGGCTGAATCTGGAGCTTTGGGGCCTGGGAACCTCTCTACCTGCGTCAATGTCTGGAGGCCCTGAGAGTTTCGCTCAGGCTCAGAGCAGGCATCGCAACCTCCCAGTTACTATTCTGTGCTGTGGCAAAGTGCCAGCTTGTCCTCTCTTCCCCACCCAGCCCGGGAAACCGGCAGCATTTCTAGTTCAGGCCCAGACCCGTCCTGGCAGCCTGGATTCCACTGCCTAGGCAGGAAGCTCATCTCAGCCCAGTGACCTT
=> 135 0-based (start: 209815997) => 209816132

In [129]:
result

,SPDI,error_info,error_valid,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,allele,info,group_name
0,NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T,NC_000001.11:230159329:CTTAAAGTGTTCAGCACTCCCCT:CT,False,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,indel,192.0,alt,NaN,GC_Mohlke


In [119]:
base_dir = '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata'
group_name_list = ['C_positive_heart_CAD', 'GC_Mendelian_variants', 'GC_Mohlke']
print([os.path.join(base_dir, f'{group_name}_indel.bed') for group_name in group_name_list])
indel_bed_file = pd.concat([pd.read_csv(os.path.join(base_dir, f'{group_name}_indel.bed'), sep="\t", header=None) for group_name in group_name_list])
indel_bed_file.columns = ['chr', 'start', 'end', 'name', 'group']

['/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/C_positive_heart_CAD_indel.bed', '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/GC_Mendelian_variants_indel.bed', '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/GC_Mohlke_indel.bed']


In [122]:
indel_bed_file[['chr', 'start', 'end', 'name']].to_csv('80K_indel_bed.bed', sep='\t', index=False, header=None)

### Solve by hand
/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/GC_Mendelian_variants/GC_Mendelian_variants.metadata.tsv.gz
/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/GC_Mohlke/GC_Mohlke.metadata.tsv.gz
/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz

- C_positive_heart_CAD
- 135 zero based (compare to the bed file)
- start + 135 => 201917640
- is the correct position: https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A201917640%2D201917650&hgsid=347309970_CdGL5A7oNaEACHjM4ebivIe8Qek2


In [125]:
# C_positive_heart_CAD
group_name = 'C_positive_heart_CAD'
current_metadata_file = '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz'
error_file_path = '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/spdi_verification/error_SPDI_C_positive_heart_CAD.tsv'

error_file = pd.read_csv(error_file_path, sep="\t", header=None)
error_file.columns = [col_SPDI, 'error_info', 'error_valid']


# concatenate metadata files:
current_metadata = read_and_eval_metadata(current_metadata_file)
# print(current_metadata.head())
current_metadata_alt = current_metadata.loc[current_metadata[col_allele].apply(hf.is_alternative)].copy()

# filter df for error spdis (left join)

# filter df for error spdis (left join)
result = pd.merge(error_file, current_metadata_alt, on=[col_SPDI], how="inner")
result.sequence.to_list()
result
# CTTCTCGGCCAATGAAGGGTCAACTCCATTGCTCTCAGCAGAACTAAATGCTTTGCCAACTGGTCTGGTGACTCATACTGGGAAGCTTGTAGCAAACCCCATTTTTGGAGAAACCGGGAAATCTCTTTGGGGAGATAAAACAAGTCACTGTAGCACTCTGCTCTCTGAAGTGCCTGGCTAGTACTTTCTGCTTCTTCATTTTATCCCAGCACCAAGGAAAACCCAAGCAAGTCTGGGAAATGTTGTAAAGATCTACACAAATGCAAGC
# 135 zero based (compare to the bed file)
# start + 135 => 201917640
# is the correct position: https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A201917640%2D201917650&hgsid=347309970_CdGL5A7oNaEACHjM4ebivIe8Qek2

,SPDI,error_info,error_valid,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,allele,info
0,NC_000001.11:201917640:T:TA,NC_000001.11:201917641:AAA:AAAA,False,C_positive_heart_CAD:ALT_rs34091558_rs34091558,CTTCTCGGCCAATGAAGGGTCAACTCCATTGCTCTCAGCAGAACTA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,201917505,201917775,.,indel,136.0,[alt],NaN


- mohlke: NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T: info: NC_000001.11:230159329:CTTAAAGTGTTCAGCACTCCCCT:CT
ref: 
- GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGCTGCACCCTTTGCTGTTGATGTGAGGCTTTATTTACCTGGGAATTGACAGGGTTTTGGTCCATGATTGCTCCTTTCTCCGCAATCACATTTAATGCCCATGATATCTCATCTTTGCACACGAAGGCAAAGCCTTCTACTGCTTTTTCTCTTAAAGTGTTCAGCACTCCCCTGAGTTTATTAGAAGGATCAGAAGAAAGAGACCCTGAGTGTGAGGCTAGGTCAAC
- alt: GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGCTGCACCCTTTGCTGTTGATGTGAGGCTTTATTTACCTGGGAATTGACAGGGTTTTGGTCCATGATTGCTCCTTTCTCCGCAATCACATTTAATGCCCATGATATCTCATCTTTGCACACGAAGGCAAAGCCTTCTACTGCTTTTTCTCTGAGTTTATTAGAAGGATCAGAAGAAAGAGACCCTGAGTGTGAGGCTAGGTCAACCGGAGAAATTCACCCAATTAT

In [55]:
test = ['/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_negative_neuron_NP.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/neuro_controls.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_positive_neuron_NP.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_positive_neuron_CD.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_positive_heart_MK.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_negative_heart_MK.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_positive_neuron_MK.metadata.tsv.gz', '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/neuro_controls/C_negative_neuron_MK.metadata.tsv.gz']
[read_and_eval_metadata(file) for file in current_metadata_file]

217
1086
99
96
97
243
100
234


[                                                  name  \
 0    C_negative_neuron_NP:GW18_PFC_ABC_chr15_894002...   
 1    C_negative_neuron_NP:NGN2_iPSC_ABC_chr4_112626...   
 2    C_negative_neuron_NP:Midfetal_Cortex_Trevino_c...   
 3    C_negative_neuron_NP:NGN2_iPSC_ABC_chr2_275913...   
 4    C_negative_neuron_NP:Fetal_Cerebrum_Cicero_chr...   
 ..                                                 ...   
 212  C_negative_neuron_NP:NGN2_iPSC_ABC_chr3_427240...   
 213  C_negative_neuron_NP:GW18_PFC_ABC_chr11_133690...   
 214  C_negative_neuron_NP:NGN2_iPSC_ABC_chr15_32862...   
 215  C_negative_neuron_NP:GW18_PFC_ABC_Midfetal_Cor...   
 216  C_negative_neuron_NP:GW18_PFC_ABC_chr4_6245249...   
 
                                               sequence category  \
 0    AGGCGCGATACGAACCCGTGGGAGCCTCCCCAACCCCGCAGTCCCA...  element   
 1    CGAAAAGTGTGTGAAGTGTGAATTATATCTCAATAAAGCTGTTAAA...  element   
 2    GGCTGAGAGGCCTGATTCCTTCCACGCATCACAACCTGAAAATCGC...  element   
 3    CATCTGTACTGA